# jaxgraphic

**Draw anything. Render it absurdly fast. In plain Python.**

Everything below runs in under a minute. Pick a GPU (*Runtime → Change runtime type*), then *Runtime → Run all*.

In [ ]:
%pip install -q git+https://github.com/EyadRealHim/jaxgraphic.git

import jax
import jax.numpy as jnp
import numpy as np
import jaxgraphic as jg

if jax.default_backend() == "cpu":
    print("You are on a CPU runtime, so everything below will be much slower.")
    print("Runtime -> Change runtime type -> GPU, then run again.")

## 1. Draw something

Shapes, colors, anti-aliasing. Five lines.

In [ ]:
from IPython.display import Image, display

(jg.Canvas.new(256, 256, "#0b1020")
   .fill(jg.Circle((128, 128), 70), "#ff5cc8", softness=1.5)
   .stroke(jg.Star((128, 128), 105, 45, points=5, angle=0.3), "#7fb8ff", width=3)
   .save("hello.png", scale=2))
display(Image("hello.png"))

## 2. Now draw a whole world

A scene is **one function that draws one frame**. This one is an aurora, fireworks and a mirror lake: stars, meteors, mountains, a rippled reflection, fireflies. *(Double-click the cell below to read all of it.)*

In [ ]:
#@title Scene: aurora, fireworks and a mirror lake { display-mode: "form" }
import math

import jax.numpy as jnp
import numpy as np

import jaxgraphic as jg

TAU = 2 * math.pi
S = 512           # canvas size in pixels
HORIZON = 300     # y of the shoreline


def hsv(h, s=0.75, v=1.0):
    """Hue in [0, 1) to RGB, channels last. Works on arrays and tracers."""
    k = lambda n: (n + 6 * h) % 6
    f = lambda n: v - v * s * jnp.clip(jnp.minimum(k(n), 4 - k(n)), 0, 1)
    return jnp.stack([f(5), f(3), f(1)], axis=-1)


# ---- fixed layout (random, but the same every frame) ---------------------------------
rng = np.random.default_rng(7)
# Stars: 22 random ones repeated at 12 rotations, so turning the sky 1/12 per loop is seamless.
n = 22
STAR_R = np.tile(380 * np.sqrt(rng.uniform(0.02, 1, n)), 12)
STAR_A = (rng.uniform(0, TAU / 12, n)[None] + TAU * np.arange(12)[:, None] / 12).ravel()
STAR_SIZE = np.tile(0.7 + 1.2 * rng.random(n) ** 3, 12)
STAR_FREQ = np.tile(rng.integers(1, 4, n), 12)
STAR_PHASE = np.tile(rng.uniform(0, TAU, n), 12)
STAR_RGB = np.tile(np.array([[1, 1, 1], [.75, .85, 1], [1, .92, .75]])[rng.integers(0, 3, n)], (12, 1))
POLE = np.array([256, 265])


def ridge(seed, base, amp):
    """Mountain outline as polygon points (closed along the shoreline)."""
    p = np.random.default_rng(seed).uniform(0, TAU, 3)
    x = np.linspace(-10, 522, 46)
    y = base - amp * (.55 * np.sin(.012 * x + p[0]) + .30 * np.sin(.031 * x + p[1])
                      + .15 * np.sin(.071 * x + p[2]) + .55)
    return np.concatenate([np.stack([x, y], -1), [[522, HORIZON + 3], [-10, HORIZON + 3]]])


FAR, NEAR = ridge(3, 290, 62), ridge(11, 298, 22)

r = np.random.default_rng(5)                             # pines: three stacked triangles each
x = np.concatenate([r.uniform(6, 150, 11), r.uniform(345, 440, 7), [506]])
h = r.uniform(46, 84, x.size)
tiers = [(HORIZON - h + i * .26 * h, h * (.14 + .065 * i)) for i in range(3)]          # (top y, half width)
TREES = np.concatenate([np.stack([(x, top), (x - hw, top + .44 * h), (x + hw, top + .44 * h)], 1).transpose(2, 1, 0)
                        for top, hw in tiers])           # (57, 3, 2)

DROP_X = np.linspace(95, 425, 7)[np.random.default_rng(2).permutation(7)]
DROP_Y = np.random.default_rng(4).uniform(335, 470, 7)

r = np.random.default_rng(9)                             # fireflies
FLY = dict(x=r.uniform(10, 500, 30), y=r.uniform(318, 478, 30), fx=r.integers(1, 4, 30), fy=r.integers(1, 4, 30),
           blink=r.integers(2, 6, 30), phase=r.uniform(0, TAU, (3, 30)))
FLY_RGB = np.where(r.random(30)[:, None] < .7, [1, .89, .48], [.78, 1, .42])

# fireworks: 6 bursts, 96 sparks each, one cycle per loop, staggered so the show never stops
r = np.random.default_rng(21)
BURST = np.array([[92, 150], [392, 128], [250, 82], [165, 182], [335, 168], [468, 120]])
BURST_HUE = np.array([.99, .12, .52, .86, .33, .60])
BURST_START = np.array([0, 3, 1, 4, 2, 5]) / 6          # when in the loop each one fires
ang = TAU * (np.arange(96)[None] + r.uniform(-.3, .3, (6, 96))) / 96
u, kind = r.random((6, 96)), np.arange(6)[:, None] % 3  # burst styles: peony, filled sphere, double ring
speed = np.where(kind == 0, .85 + .15 * u, np.where(kind == 1, np.sqrt(u), np.where(u < .5, .55, 1) + .04 * u))
SPARK_DIR = np.stack([np.cos(ang), np.sin(ang)], -1) * speed[..., None]
SPARK_HUE = r.uniform(-.04, .04, (6, 96))
SPARK_FREQ, SPARK_PHASE = r.integers(18, 41, (6, 96)), r.uniform(0, TAU, (6, 96))
LAUNCH = .2                                              # fraction of a cycle the rocket spends climbing

# (x, y, heading, length, travel, start time, duration)
METEORS = np.array([[120, 30, .50, 62, 210, .10, .15],
                    [470, 58, math.pi - .62, 52, 170, .58, .12]])


# ---- whole-image passes (applied with Canvas.pipe) -----------------------------------
def aurora(img, a, gain):
    """Three curtains of additive light: bright lower edge, rays fading upward, green to violet."""
    x, y = jnp.arange(S)[None, :] + .5, jnp.arange(S)[:, None] + .5
    for j in range(3):
        mid = (118 + 36 * j + 24 * jnp.sin(.020 * (1 + .25 * j) * x + (j + 1) * a + j)
               + 12 * jnp.sin(.052 * x - 2 * a + 2 * j))
        d = (y - mid) / (30 + 9 * j)
        glow = jnp.exp(-jnp.where(d > 0, 5 * d * d, .35 * d * d))
        rays = .5 + .5 * jnp.sin(.27 * x + 2.2 * jnp.sin(.034 * x + (j + 1) * a) + 1.7 * j)
        patch = .5 + .5 * jnp.sin(.0125 * (1 + .2 * j) * x + 1.9 * j + a)
        light = glow * (.12 + .88 * rays ** 3) * (.1 + .9 * patch ** 2)
        colour = jnp.moveaxis(hsv(.36 + .40 * jnp.clip(.12 - .5 * d, 0, 1) + .03 * j, .7), -1, 0)
        img = img + gain * (.85 - .12 * j) * light * colour
    return jnp.clip(img, 0, 1)


def water(img, a):
    """Everything below the shoreline becomes a rippled, fading mirror of everything above it."""
    n = S - HORIZON
    d = (jnp.arange(n) / n)[:, None]                                   # 0 at the shore, 1 at the bottom edge
    up = img[:, HORIZON - 1 - np.floor(np.arange(n) * 1.25).astype(int)]  # mirrored rows, a little compressed
    x = jnp.arange(S)[None, :]
    shift = (.6 + 4.5 * d) * jnp.sin(30 * d - 2 * a + .03 * (x + .5))    # horizontal wobble, in pixels
    xs = jnp.clip(x + shift, 0, S - 1.001)
    x0 = jnp.floor(xs).astype(jnp.int32)
    pick = lambda i: jnp.take_along_axis(up, jnp.broadcast_to(i, (3, n, S)), axis=2)
    frac = xs - x0
    mirror = pick(x0) * (1 - frac) + pick(jnp.minimum(x0 + 1, S - 1)) * frac          # bilinear
    mirror = mirror * jnp.array([.80, .92, 1.])[:, None, None]                        # cool tint
    near, far = jg.rgb("#0e3b4b"), jg.rgb("#040c1e")
    base = near[:, None, None] + d * (far - near)[:, None, None]
    return jnp.concatenate([img[:, :HORIZON], base + (.80 - .45 * d) * (mirror - base)], axis=1)


def vignette(img):
    x, y = jnp.arange(S)[None, :] + .5, jnp.arange(S)[:, None] + .5
    return img * (1 - .5 * (((x - S / 2) ** 2 + (y - S / 2) ** 2) / (S * S / 2)) ** 1.3)


# ---- layers --------------------------------------------------------------------------
def sky(a):
    cv = (jg.Canvas.new(S, S, "#03051a")
          .gradient(jg.Box((256, 256), (S, S)), (0, 0), (0, HORIZON), "#03051a", "#0f4a5e")
          .fill(jg.Box((256, 110), (780, 80), angle=-.55), "#8fa2ff", alpha=.10, softness=95)      # Milky Way
          .fill(jg.Box((256, 110), (700, 22), angle=-.55), "#dfe6ff", alpha=.07, softness=40))
    angle = STAR_A + a / 12
    pos = POLE + STAR_R[:, None] * jnp.stack([jnp.cos(angle), jnp.sin(angle)], -1)
    twinkle = .5 + .5 * jnp.sin(STAR_FREQ * a + STAR_PHASE)
    return cv.fill_many(jg.Circle(pos, STAR_SIZE), STAR_RGB, .45 + .55 * twinkle ** 1.5,
                        softness=1.6, at=pos, window=(8, 8))


def meteors(cv, t):
    x, y, heading, length, travel, t0, dur = METEORS.T
    q = jnp.clip((t - t0) / dur, 0, 1)
    way = np.stack([np.cos(heading), np.sin(heading)], -1)
    head = np.stack([x, y], -1) + way * (travel * q)[:, None]
    tail = head - way * length[:, None]
    alpha, mid = jnp.sin(math.pi * q), (head + tail) / 2
    streak = lambda width: jg.Segment(tail, head, jnp.full(2, width))
    return (cv.fill_many(streak(5.), ["#7fb8ff"] * 2, .35 * alpha, softness=7, at=mid, window=(82, 58))
              .fill_many(streak(1.8), ["#f2f8ff"] * 2, alpha, softness=1.4, at=mid, window=(82, 58)))


def moon(cv, a):
    m, R = jnp.array([410., 82.]), 26
    shift = R * (.45 + .9 * (.5 - .5 * jnp.cos(a)))                    # the crescent breathes
    lit = jg.Circle(m, R) - jg.Circle(m + jnp.array([shift, -3.]), R)
    return (cv.fill(jg.Circle(m, R), "#9fd0ff", alpha=.28, softness=150)
              .fill(jg.Circle(m, R), "#5a7096", alpha=.45)             # earthshine
              .fill(lit, "#fff3d6", softness=1.3))


def fireworks(cv, t, a):
    phase = (t + BURST_START) % 1                                      # (6,) where each burst is in its cycle
    # rockets climbing from the shore
    s = jnp.clip(phase / LAUNCH, 0, 1)
    ground = HORIZON - 6
    head = jnp.stack([BURST[:, 0], ground + (BURST[:, 1] - ground) * (1 - (1 - s) ** 2.2)], -1)
    tail, up = head + jnp.array([0., 24.]), jnp.where(phase < LAUNCH, 1., 0.)
    mid = (head + tail) / 2
    rocket = lambda width: jg.Segment(tail, head, jnp.full(6, width))
    cv = (cv.fill_many(rocket(5.), ["#ffb347"] * 6, .35 * up, softness=8, at=mid, window=(22, 38))
            .fill_many(rocket(1.8), ["#fff1d6"] * 6, up, softness=1.3, at=mid, window=(22, 38)))
    # flash at the moment of the burst
    flash = jnp.clip(1 - jnp.where(phase >= LAUNCH, phase - LAUNCH, 1) / .07, 0, 1)
    cv = cv.fill_many(jg.Circle(BURST, 12 + 34 * (1 - flash)), ["#fff0d0"] * 6, .9 * flash,
                      softness=60, at=BURST, window=(152, 152))
    # sparks: short streaks from where each one was a moment ago to where it is now
    tau = jnp.clip((phase - LAUNCH) / (1 - LAUNCH), 0, 1)[:, None]      # 0 at the burst, 1 when it has faded

    def where(q):                                                       # drag-limited expansion plus gravity
        out = (1 - jnp.exp(-3.5 * q)) / (1 - math.exp(-3.5))
        return BURST[:, None] + 112 * SPARK_DIR * out[..., None] + jnp.stack([jnp.zeros_like(q), 70 * q * q], -1)

    p1, p0 = where(tau).reshape(-1, 2), where(jnp.clip(tau - .06, 0, 1)).reshape(-1, 2)
    flicker = .65 + .35 * jnp.sin(SPARK_FREQ * a + SPARK_PHASE)
    fade = (jnp.where(phase[:, None] >= LAUNCH, 1., 0.) * (1 - tau) ** 1.05 * flicker).reshape(-1)
    colour = hsv(BURST_HUE[:, None] + SPARK_HUE, jnp.clip(.12 + 3 * tau, 0, .85)).reshape(-1, 3)
    life, mid = jnp.broadcast_to(tau, (6, 96)).reshape(-1), (p0 + p1) / 2
    cv = cv.fill_many(jg.Segment(p0, p1, 6 + 3 * (1 - life)), colour, .55 * fade,
                      softness=13, at=mid, window=(68, 68))
    return cv.fill_many(jg.Segment(p0, p1, 2.4 + 1.6 * (1 - life)), .4 * colour + .6, fade,
                        softness=1.3, at=mid, window=(68, 68))


def land(cv, a):
    cv = (cv.fill(jg.Polygon(FAR), "#1a2c52")
            .fill(jg.Box((256, 290), (540, 46)), "#3fb0a6", alpha=.30, softness=60)      # mist
            .fill(jg.Polygon(NEAR), "#0a1430")
            .fill_many(jg.Polygon(TREES), ["#040813"] * len(TREES), 1., softness=1.,
                       at=(TREES.min(1) + TREES.max(1)) / 2, window=(54, 44)))
    flicker = .85 + .15 * jnp.sin(5 * a + 1) * jnp.sin(3 * a)                            # cabin window
    return (cv.fill(jg.Box((470, 291), (30, 18)), "#070c1c")
              .fill(jg.Triangle((451, 282), (489, 282), (470, 268)), "#070c1c")
              .fill(jg.Circle((466, 291), 5), "#ff9f43", alpha=.55 * flicker, softness=34)
              .fill(jg.Box((466, 291), (6, 6)), "#ffc766", alpha=flicker))


def lake_life(cv, t, a):
    # ripple rings: one drop per ring, each expanding once per half loop
    age = (2 * t + np.arange(7) / 7) % 1
    drop = np.stack([DROP_X, DROP_Y], -1)
    r = 6 + 56 * age
    rings = jg.Outline(jg.Ellipse(drop, jnp.stack([r, r * .26], -1)), 1.8 - .9 * age)
    cv = cv.fill_many(rings, ["#bfe9ff"] * 7, .7 * (1 - age) ** 1.5, softness=1.3, at=drop, window=(132, 42))
    # fireflies: a soft glow, then a bright core
    f = FLY
    pos = jnp.stack([f["x"] + 16 * jnp.sin(f["fx"] * a + f["phase"][0]),
                     f["y"] + 10 * jnp.sin(f["fy"] * a + f["phase"][1])], -1)
    blink = jnp.clip(.5 + .9 * jnp.sin(f["blink"] * a + f["phase"][2]), 0, 1) ** 2
    dots = jg.Circle(pos, jnp.full(30, 1.2))
    cv = cv.fill_many(dots, FLY_RGB, .65 * blink, softness=13, at=pos, window=(18, 18))
    return cv.fill_many(dots, .5 * FLY_RGB + .5, blink, softness=1.3, at=pos, window=(18, 18))


def scene(t, finale=True):
    """One frame. t in [0, 1); everything moves a whole number of turns per loop, so it loops."""
    a = TAU * t
    cv = moon(meteors(sky(a), t), a).pipe(aurora, a, .55 if finale else 1.)
    if finale:
        cv = fireworks(cv, t, a)
    cv = land(cv, a).pipe(water, a)
    return lake_life(cv, t, a).pipe(vignette)

In [ ]:
#@title Helpers for the results { display-mode: "form" }
from IPython.display import HTML, display


def count_shapes():
    """Shapes drawn in one frame, counted while tracing (nothing is rendered)."""
    n, orig = [0], {k: getattr(jg.Canvas, k) for k in ("fill", "stroke", "gradient", "fill_many")}
    def wrap(name):
        def method(self, *args, **kw):
            n[0] += jax.tree_util.tree_leaves(args[0])[0].shape[0] if name == "fill_many" else 1
            return orig[name](self, *args, **kw)
        return method
    for k in orig:
        setattr(jg.Canvas, k, wrap(k))
    try:
        jax.eval_shape(lambda t: scene(t).img, 0.3)
    finally:
        for k, f in orig.items():
            setattr(jg.Canvas, k, f)
    return n[0]


def reveal(video, took):
    """Big-number cards, all computed from this run."""
    n = len(video)
    fps, secs = n / took, n / 30                       # secs of video at 30 fps
    ratio = secs / took
    shapes = count_shapes()
    film = 24 * 7200 / fps                             # a 2-hour film at 24 fps
    nice = lambda s: f"{s:.0f} s" if s < 120 else f"{s / 60:.1f} min"
    cards = [(f"{ratio:,.0f}x" if ratio >= 10 else f"{ratio:.1f}x", "real time",
              f"{secs:.0f} seconds of video rendered in {took:.2f} seconds"),
             (f"{fps * 0.1:,.0f}", "frames in one blink", "a blink lasts about a tenth of a second"),
             (f"{shapes:,}", "shapes in every frame", f"{shapes * fps / 1e6:.1f} million shapes drawn per second"),
             (nice(film), "for a 2-hour film", "at this speed: this scene, 24 fps")]
    box = ("flex:1 1 200px;background:#0b1020;border-radius:14px;padding:18px 20px;"
           "font-family:system-ui,sans-serif;color:#e8ecf7;min-width:180px")
    html = "".join(f'<div style="{box}"><div style="font-size:42px;font-weight:700;line-height:1.1;color:#7fd8ff">{big}</div>'
                   f'<div style="font-size:16px;font-weight:600;margin-top:4px">{label}</div>'
                   f'<div style="font-size:13px;opacity:.65;margin-top:6px">{small}</div></div>' for big, label, small in cards)
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:12px;margin:8px 0">{html}</div>'))

## 3. Render the whole film

`jax.lax.map` renders 24 frames per call, and frames are converted to `uint8` on the GPU.

In [ ]:
import time

FRAMES = 240 if jax.default_backend() != "cpu" else 24         # 8 seconds at 30 fps

frame = lambda t: jg.to_uint8(scene(t).img)
render = jax.jit(lambda ts: jax.lax.map(frame, ts, batch_size=24))
ts = jnp.linspace(0, 1, FRAMES, endpoint=False)

print("Compiling (happens once)...", flush=True)
render(ts).block_until_ready()

runs = []
for _ in range(3):
    t0 = time.perf_counter()
    video = render(ts).block_until_ready()
    runs.append(time.perf_counter() - t0)

reveal(video, float(np.median(runs)))

In [ ]:
#@title Watch it { display-mode: "form" }
import base64
import imageio.v2 as imageio

imageio.mimsave("lake.mp4", list(np.asarray(video).transpose(0, 2, 3, 1)), fps=30, macro_block_size=1)
b64 = base64.b64encode(open("lake.mp4", "rb").read()).decode()
HTML(f'<video src="data:video/mp4;base64,{b64}" width="512" autoplay loop muted controls></video>')

You just watched eight seconds of video that took a fraction of a second to make.

## Why it's fast

- Every shape is a JAX pytree, so `jax.jit` fuses the whole scene into one GPU program.
- `lax.map` / `jax.vmap` turn the one-frame function into a many-frames function. There is nothing to learn.
- `fill_many(..., window=...)` draws thousands of stars and sparks that each touch only a tiny patch of the image.

## Make it yours

Edit the scene cell above and run again. Try `scene(t, finale=False)` for the calm version, change a color, or move the moon.

[Read the docs](https://github.com/EyadRealHim/jaxgraphic#readme) · [Star the repo](https://github.com/EyadRealHim/jaxgraphic)